# Chapter 3 companion: discrete-time systems and the z-transform.

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Chapters 1–11 of the notes are based on the lecture notes of Prof. Mario Sznaier; all credits are in `ATTRIBUTION.md`.

Reproduces:
  * the satellite (unit inertia, torque through a sample-and-hold): the
    difference equation theta(k+2) - 2 theta(k+1) + theta(k) = T^2/2 [tau(k+1) + tau(k)],
    checked against the exact solution theta(t) = t^2/2 for a constant torque,
    and the pulse transfer function T^2 (z + 1) / (2 (z - 1)^2);
  * the definition E(z) = sum e_k z^-k: partial sums against the closed forms
    of a few pairs, inside and outside the region of convergence;
  * the final value theorem on the position loop with F(z) = (0.368z + 0.264)/((z-1)(z-0.368)):
    the error for K = 1 (stable, e -> 0) and K = 3 (unstable, the formula still says 0).

Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import signal

FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)

In [ ]:
# ---------------------------------------------------------------- satellite
T = 1.0

In [ ]:
n = 8
tau = np.ones(n + 2)                 # constant torque from k = 0
theta = np.zeros(n + 2)
omega = np.zeros(n + 2)

In [ ]:
for k in range(n + 1):               # exact integration over each period (torque held)
    theta[k + 1] = theta[k] + T * omega[k] + T**2 / 2 * tau[k]
    omega[k + 1] = omega[k] + T * tau[k]

In [ ]:
print("Satellite, T = 1 s, unit torque from rest")
print("  theta(k)          :", theta[:n])
print("  k^2/2             :", 0.5 * np.arange(n) ** 2)
lhs = theta[2:n] - 2 * theta[1:n - 1] + theta[:n - 2]
rhs = T**2 / 2 * (tau[1:n - 1] + tau[:n - 2])
print("  second-order difference equation holds:", np.allclose(lhs, rhs))

In [ ]:
# the same samples from the pulse transfer function T^2 (z+1) / (2 (z-1)^2), driven by a step
num, den = [0, T**2 / 2, T**2 / 2], np.polymul([1, -1], [1, -1])

In [ ]:
print("  Theta/Tau = (T^2/2)(z+1)/(z-1)^2, step response:", signal.lfilter(num, den, np.ones(n)))
print("  T = 0.5 s: T^2/2 =", 0.5**2 / 2)

In [ ]:
# ---------------------------------------------------------------- definition and ROC
print("\nPartial sums of sum e_k z^-k (200 terms) against the closed form")

In [ ]:
k = np.arange(200)

In [ ]:
for name, ek, Ez, roc in [("a^k, a = 0.5", 0.5**k, lambda z: z / (z - 0.5), 0.5),
                          ("(-0.8)^k", (-0.8) ** k, lambda z: z / (z + 0.8), 0.8),
                          ("unit step", np.ones(200), lambda z: z / (z - 1), 1.0)]:
    for z in (1.5, 2.0 * np.exp(0.7j)):
        s = np.sum(ek * z ** (-k.astype(float)))
        print(f"  {name:12s} z = {complex(z):.3f}: series {complex(s):.6f}  closed form {complex(Ez(z)):.6f}"
              f"   (ROC |z| > {roc})")

In [ ]:
z = 0.6
print(f"  0.5^k at z = 0.6 (inside the ROC): partial sums 50, 100, 200 terms:",
      [round(float(np.sum(0.5**k[:N] * z ** (-k[:N].astype(float)))), 4) for N in (50, 100, 200)],
      " closed form", round(z / (z - 0.5), 4))
z = 0.4
print(f"  0.5^k at z = 0.4 (outside the ROC): partial sums 10, 20, 40 terms:",
      [f"{np.sum(0.5**k[:N] * z ** (-k[:N].astype(float))):.3g}" for N in (10, 20, 40)], " (diverge)")

In [ ]:
# ---------------------------------------------------------------- scaling and shifts
z = 1.7 + 0.4j

In [ ]:
E = lambda z: z / (z - 0.5)
print("\nShift rules for e_k = 0.5^k at z =", z)
print("  Z[e(k-3)]  =", np.round(z**-3 * E(z), 6), " closed form 1/(z^2(z-0.5)) =", np.round(1 / (z**2 * (z - 0.5)), 6))
print("  Z[e(k+2)]  =", np.round(z**2 * (E(z) - 1 - 0.5 / z), 6), " = 0.25 z/(z-0.5) =", np.round(0.25 * z / (z - 0.5), 6))
print("  Z[(-0.5)^k] by scaling, E(-z) =", np.round(E(-z), 6), " = z/(z+0.5) =", np.round(z / (z + 0.5), 6))

In [ ]:
# ---------------------------------------------------------------- position loop
e1 = np.exp(-1.0)                                     # exact: 0.368 = e^-1, 0.264 = 1 - 2e^-1

In [ ]:
Fn = [e1, 1 - 2 * e1]
Fd = np.polymul([1, -1], [1, -e1])
print("\nPosition loop, F(z) = (0.368z + 0.264)/((z-1)(z-0.368)) (exact e^-1 used), unit step reference")
fig, ax = plt.subplots(figsize=(6.4, 3.4))

In [ ]:
for K, c in [(1, "C0"), (3, "C3")]:
    char = np.polyadd(Fd, K * np.array([0, *Fn]))       # (z-1)(z-0.368) + K(0.368z + 0.264)
    poles = np.roots(char)
    e = signal.lfilter(Fd, char, np.ones(16))            # E/R = 1/(1 + K F)
    print(f"  K = {K}: characteristic polynomial {np.round(char, 3)}, poles {np.round(poles, 3)},"
          f" |z| = {abs(poles[0]):.3f}")
    print(f"         e(k) = {np.round(e[:10], 3)}")
    ax.plot(np.arange(16), e, "o-", ms=3, color=c, label=f"K = {K}")

In [ ]:
print("  final value formula: lim_{z->1} z/(1 + K F(z)) = 0 for every K (F has a pole at z = 1);")
print("  it is the limit of e(k) only when the loop is stable (0 < K < 2.39).")
ax.axhline(0, color="k", lw=0.5)
ax.set(xlabel="k", ylabel="e(k)", title="Position loop: error samples")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "ch03_position_error.png", dpi=150)
print("\nSaved figures/ch03_position_error.png")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**The pole of $a^k$ and where its series converges.** The sequence $e_k = a^k$ has
$E(z) = \sum_k a^k z^{-k} = z/(z-a)$, but only for $|z| > |a|$, the exterior of the disk
through the pole. Move $a$ and watch the sequence: it decays when the pole is inside the unit
circle and grows when it is outside. Then move the test point $z$ inside the disk: the closed
form still returns a number (try $a = 0.5$, $z = 0.25$: it says $-1$), but the partial sums
run away.
"""
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"
NK = 21        # samples shown, k = 0..20


def explore(a=0.5, z=1.0):
    a, z = round(a, 10), round(z, 10)        # slider values such as 0.49999999999
    k = np.arange(NK)
    e = a ** k
    on_pole = abs(z - a) < 1e-12
    inside_roc = abs(z) > abs(a)
    closed = np.inf if on_pole else z / (z - a)
    partial = np.cumsum(e * z ** (-k.astype(float)))

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12.5, 3.9),
                                        gridspec_kw={"width_ratios": [1.25, 1, 1.25]})
    ml, sl, bl = ax1.stem(k, e)
    plt.setp(ml, color=ORANGE, markersize=5)
    plt.setp(sl, color=ORANGE)
    plt.setp(bl, color="k", lw=0.6)
    ax1.set_xlabel("k")
    ax1.set_ylabel(r"$e_k = a^k$")
    ax1.set_title(f"a = {a:g}: " + ("decays" if abs(a) < 1 else
                                    "stays bounded" if abs(a) == 1 else "grows"), fontsize=11)

    lim = 1.6
    th = np.linspace(0, 2 * np.pi, 300)
    ax2.add_patch(plt.Rectangle((-lim, -lim), 2 * lim, 2 * lim, color=TEAL, alpha=0.15, lw=0))
    ax2.add_patch(plt.Circle((0, 0), abs(a), color="white", lw=0))
    ax2.plot(abs(a) * np.cos(th), abs(a) * np.sin(th), color=TEAL, lw=1.2)
    ax2.plot(np.cos(th), np.sin(th), color=GRAY, lw=1, ls="--")
    ax2.axhline(0, color="k", lw=0.5)
    ax2.axvline(0, color="k", lw=0.5)
    ax2.plot([a], [0], "x", color=ORANGE, ms=12, mew=2.5, label="pole z = a")
    ax2.plot([z], [0], "o", color=BLUE, ms=8, label="test point z")
    ax2.set_xlim(-lim, lim)
    ax2.set_ylim(-lim, lim)
    ax2.set_aspect("equal")
    ax2.set_xlabel("Re z")
    ax2.set_ylabel("Im z")
    ax2.set_title("ROC |z| > |a| (shaded)", fontsize=11)
    ax2.legend(fontsize=10, loc="upper left", framealpha=0.9)

    ax3.plot(k, partial, "o-", color=BLUE, ms=4, lw=1.2, label="partial sums")
    if np.isfinite(closed):
        ax3.axhline(closed, color=ORANGE, ls="--", lw=2, label=f"z/(z-a) = {closed:.3g}")
    ax3.set_xlabel("k (last term in the sum)")
    ax3.set_ylabel(r"$\sum_{i\leq k} a^i z^{-i}$")
    ax3.set_title(f"series at z = {z:g}: " + ("converges" if inside_roc else "diverges"),
                  fontsize=11)
    if not inside_roc:                      # the sums leave the window: that is the point
        c = closed if np.isfinite(closed) else 0.0
        ax3.set_ylim(c - 6, c + 6)
    ax3.legend(fontsize=10, loc="upper left")
    fig.tight_layout()

    print(f"pole z = a = {a:g}; ROC |z| > {abs(a):g}; "
          + ("pole inside the unit circle: a^k -> 0" if abs(a) < 1 else
             "pole on the unit circle: a^k does not decay" if abs(a) == 1 else
             "pole outside the unit circle: a^k grows"))
    if on_pole:
        print(f"z = {z:g} sits on the pole: z/(z-a) is not defined")
    else:
        print(f"closed form z/(z-a) at z = {z:g}: {closed:.4f}")
    print(f"partial sum with {NK} terms: {partial[-1]:.4g}  ("
          + ("inside the ROC, it approaches the closed form)" if inside_roc else
             "outside the ROC, the series has no sum)"))
    plt.show()


try:
    from ipywidgets import interact, FloatSlider
    interact(explore,
             a=FloatSlider(value=0.5, min=-1.2, max=1.2, step=0.05, description="pole a",
                           continuous_update=False),
             z=FloatSlider(value=1.0, min=0.1, max=1.6, step=0.05, description="test z",
                           continuous_update=False))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 3.** These cells go with the Hands-on exercises at the end of
Chapter 3. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
def first_order_step(p, n):
    """x(k+1) = p x(k) + (1 - p) u(k), u = 1, from rest: samples x(0..n-1)."""
    x = np.zeros(n)
    for k in range(n - 1):
        x[k + 1] = p * x[k] + (1 - p)
    return x

In [ ]:
def settle_index(x, tol=0.02):
    """first k after which |x - 1| <= tol for good."""
    bad = np.nonzero(np.abs(x - 1) > tol)[0]
    return 0 if len(bad) == 0 else bad[-1] + 1

In [ ]:
# ---------------------------------------------------------------- H1
def h1_pole_and_T(T_motor_ms=1.0, T_green_s=60.0):
    """H1: motor (time constant 0.0995 s) and greenhouse (86.6 s); same law x+ = p x + (1-p) u."""
    tau_m, tau_g = -1e-3 / np.log(0.99), -60 / np.log(0.5)
    pm, pg = np.exp(-T_motor_ms * 1e-3 / tau_m), np.exp(-T_green_s / tau_g)
    nm = int(np.ceil(0.8 / (T_motor_ms * 1e-3))) + 1
    ng = int(np.ceil(600 / T_green_s)) + 1
    xm, xg = first_order_step(pm, nm), first_order_step(pg, ng)
    km, kg = settle_index(xm), settle_index(xg)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8))
    ax1.plot(np.arange(nm), xm, ".", color=ORANGE, ms=3, label=f"motor, p = {pm:.5f}")
    ax1.plot(np.arange(ng), xg, "o", color=BLUE, ms=5, label=f"greenhouse, p = {pg:.3f}")
    ax1.set_xscale("symlog", linthresh=10)
    ax1.set_xlabel("sample k")
    ax1.set_title(f"in samples: motor {km}, greenhouse {kg}", fontsize=11)
    ax1.legend(loc="lower right", fontsize=9)
    # log time axis: start at k = 1 (t = 0 cannot be drawn)
    ax2.plot(np.arange(1, nm) * T_motor_ms * 1e-3, xm[1:], color=ORANGE, lw=1.6, label="motor")
    ax2.plot(np.arange(1, ng) * T_green_s, xg[1:], "o-", color=BLUE, ms=4, lw=1,
             label="greenhouse")
    ax2.set_xscale("log")
    ax2.set_xlim(1e-3, 700)
    ax2.set_xlabel("t (s)")
    ax2.set_title(f"in seconds: motor {km * T_motor_ms * 1e-3:.3f} s, "
                  f"greenhouse {kg * T_green_s:.0f} s", fontsize=11)
    ax2.legend(loc="lower right", fontsize=9)
    for ax in (ax1, ax2):
        ax.axhspan(0.98, 1.02, color=TEAL, alpha=0.15)
        ax.set_ylim(0, 1.1)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_insulin(q0=2.0, basal=0.5):
    """H2: q(k+1) = 0.8 q(k) + u(k); the recursion against the flawed formula 2.5 (1 - 0.8^k)."""
    n = 25
    q = np.zeros(n)
    q[0] = q0
    for k in range(n - 1):
        q[k + 1] = 0.8 * q[k] + basal
    k = np.arange(n)
    flawed = basal / 0.2 * (1 - 0.8 ** k)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(k, q, "o-", color=ORANGE, ms=5, lw=1, label="recursion q(k+1) = 0.8 q(k) + u(k)")
    ax.plot(k, flawed, "s--", color=TEAL, ms=4, lw=1, label="classmate's q(k)")
    ax.axhline(basal / 0.2, color=GRAY, lw=0.8)
    ax.set_xlabel("k (periods of 5 min)")
    ax.set_ylabel("insulin on board (U)")
    ax.set_title(f"q(0) = {q0:g} U, basal {basal:g} U: q(1) = {q[1]:.3f}, q(2) = {q[2]:.3f}, "
                 f"final {basal / 0.2:.2f} U", fontsize=11)
    ax.legend(loc="lower right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_hum(N=5):
    """H3: weigh scale at 600 Hz, 1 kg step at k = 0 plus 0.2 kg of 60 Hz hum; average of N samples."""
    T = 1 / 600
    k = np.arange(-20, 80)
    e = np.where(k >= 0, 1.0, 0.0) + 0.2 * np.cos(2 * np.pi * 60 * k * T)
    c = np.array([e[i - N + 1:i + 1].mean() for i in range(N - 1, len(k))])
    kc = k[N - 1:]
    late = c[kc >= max(N - 1, 0) + 20]
    ripple = (late.max() - late.min()) / 2
    zeros = np.exp(2j * np.pi * np.arange(1, N) / N)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [2, 1]})
    ax1.plot(k * T * 1000, e, color=GRAY, lw=1, label="raw reading")
    ax1.plot(kc * T * 1000, c, ".-", color=ORANGE, ms=4, lw=1.2, label=f"average of {N}")
    ax1.axvline((N - 1) * T * 1000, color=BLUE, ls=":", lw=1)
    ax1.set_xlabel("t (ms)")
    ax1.set_ylabel("kg")
    ax1.set_title(f"N = {N}: remaining hum amplitude {ripple:.4f} kg", fontsize=11)
    ax1.legend(loc="lower right", fontsize=9)
    th = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(th), np.sin(th), color=GRAY, lw=1)
    ax2.plot(zeros.real, zeros.imag, "o", mfc="none", color=BLUE, ms=8, label="zeros of H")
    ax2.plot(0, 0, "x", color=ORANGE, ms=10, mew=2, label=f"{N - 1} poles")
    hum = np.exp(1j * np.pi / 5 * np.array([1, -1]))
    ax2.plot(hum.real, hum.imag, "*", color=TEAL, ms=12, label="hum")
    ax2.set_aspect("equal")
    ax2.set_xlim(-1.4, 1.4)
    ax2.set_ylim(-1.3, 1.3)
    ax2.legend(loc="upper left", fontsize=7)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_gimbal(T=0.25, A=0.032, pattern="A, -A"):
    """H4: J theta'' = tau, torque held over each period; pulse pattern, then zero torque."""
    J = 0.02
    signs = {"A, -A": [1, -1], "A, 0, -A": [1, 0, -1], "A, A, -A, -A": [1, 1, -1, -1]}[pattern]
    taus = A * np.array(signs + [0, 0, 0])
    t_all, th_all = [], []
    th, w = 0.0, 0.0
    for k, tau in enumerate(taus):
        s = np.linspace(0, T, 40)
        t_all.append(k * T + s)
        th_all.append(th + w * s + tau / J * s ** 2 / 2)
        th, w = th + w * T + tau / J * T ** 2 / 2, w + tau / J * T
    t_all, th_all = np.concatenate(t_all), np.concatenate(th_all)
    n_move = len(signs)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(t_all, th_all, color=ORANGE, lw=1.6, label="angle between samples")
    ks = np.arange(len(taus) + 1)
    ax.plot(ks * T, np.interp(ks * T, t_all, th_all), "o", color=BLUE, ms=6, label="samples")
    ax.axhline(0.1, color=TEAL, ls="--", lw=1, label="target 0.1 rad")
    ax.axvline(0.5, color=GRAY, ls=":", lw=1, label="t = 0.5 s")
    ax.set_xlabel("t (s)")
    ax.set_ylabel("$\\theta$ (rad)")
    ok = lambda c: "pass" if c else "FAIL"
    ax.set_title(f"T = {T:g} s, A = {A:.4f} N m: final {th:.4f} rad, velocity {w:.2e}; "
                 f"torque {ok(abs(A) <= 0.04 + 1e-12)}, time {n_move * T:.3f} s "
                 f"({ok(n_move * T <= 0.5 + 1e-12)})", fontsize=10)
    ax.legend(loc="lower right", fontsize=9)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    interact(h1_pole_and_T,
             T_motor_ms=FloatSlider(value=1.0, min=0.1, max=20, step=0.1, description="T motor (ms)",
                                    continuous_update=False),
             T_green_s=FloatSlider(value=60, min=10, max=300, step=10, description="T green (s)",
                                   continuous_update=False))
    interact(h2_insulin,
             q0=FloatSlider(value=2.0, min=0, max=4, step=0.1, description="q(0) (U)",
                            continuous_update=False),
             basal=FloatSlider(value=0.5, min=0, max=1, step=0.05, description="basal (U)",
                               continuous_update=False))
    interact(h3_hum, N=IntSlider(value=5, min=1, max=30, description="N", continuous_update=False))
    interact(h4_gimbal,
             T=FloatSlider(value=0.25, min=0.05, max=0.4, step=0.005, description="T (s)",
                           continuous_update=False),
             A=FloatSlider(value=0.032, min=0.0, max=0.06, step=0.001, description="A (N m)",
                           readout_format=".3f", continuous_update=False),
             pattern=Dropdown(options=["A, -A", "A, 0, -A", "A, A, -A, -A"], value="A, -A",
                              description="pulses"))
except ImportError:
    h1_pole_and_T()
    h2_insulin()
    h3_hum()
    h4_gimbal()